<a id="s1-top"></a>
<div style="font-family: 'Georgia', serif; max-width: 900px; margin: 0 auto; padding: 40px; background-color: #1a1a1a; color: #e0e0e0; border-top: 4px solid #d4af37; border-bottom: 2px solid #333;">
    <div style="font-size: 12px; letter-spacing: 4px; color: #9b2226; text-transform: uppercase; font-weight: bold;">Lab 01 // Sprint 01</div>
    <h1 style="font-weight: 400; font-size: 48px; margin: 10px 0 5px; line-height: 1.1; color: #e0e0e0;">Случайные величины<br>в морфометрии макак</h1>
    <div style="font-size: 16px; color: #888; font-style: italic; margin-bottom: 20px;">описательная статистика, EDA, проверка гипотез</div>
    <div style="height: 1px; background: #333; margin: 20px 0;"></div>
    <div style="font-size: 12px; color: #888; letter-spacing: 1px;">
        КТвД &nbsp;·&nbsp; XeroX-cat &nbsp;·&nbsp; <a href="https://github.com/XeroX-cat/1Laba" style="color: #d4af37; text-decoration: none;">github.com/XeroX-cat/1Laba</a>
    </div>
</div>

### Индекс разделов
1. [Подготовка датасета](#s1-dataset)
2. [Природа события](#s1-nature)
3. [Величина 01: Масса](#s1-mass)
4. [Величина 02: Длина тела](#s1-blength)
5. [Величина 03: Плечевая кость](#s1-humerus)
6. [Величина 04: Бедренная кость](#s1-femur)
7. [Зависимости и нормализация](#s1-deps)
8. [Выводы](#s1-resume)

<a id="s1-dataset"></a>
<div style="font-family: 'Georgia', serif; border-left: 4px solid #d4af37; padding-left: 20px; margin: 40px 0 20px;">
    <div style="font-size: 12px; letter-spacing: 2px; color: #888;">РАЗДЕЛ 1</div>
    <h2 style="margin: 5px 0 0; color: #e0e0e0; font-weight: 400;">Подготовка датасета</h2>
</div>

Источник: [PMC3403949](https://pmc.ncbi.nlm.nih.gov/articles/PMC3403949/). Загружаем `Макаки.xlsx`, оставляем пол, долю китайского происхождения, массу тела, длину тела, длину плечевой и бедренной костей; строки с пропусками удаляем.

<details>
<summary style="cursor: pointer; color: #d4af37; font-weight: bold;">[ Показать/Скрыть код ]</summary>

Код ниже выполняет загрузку и очистку. Результаты выводятся под ячейками.
</details>

In [ ]:
# %pip install pandas xlrd
import pandas as pd

toque = pd.read_excel("Макаки.xlsx")
toque.head()

In [ ]:
toque.info()

In [ ]:
selected = toque[["Sex", "Chinese Ancestry", "kg", "C-R curved", "R humeral max", "R femoral max"]].copy()
selected.isna().sum()

In [ ]:
complete = selected.dropna()
print("Всего строк:", len(selected))
print("Строк со всеми тремя измерениями:", len(complete))
print("Строк хотя бы с одним пропуском:", len(selected) - len(complete))

In [ ]:
sample_with = complete.sample(n=50, replace=True, random_state=42)
sample_without = complete.sample(n=50, replace=False, random_state=43)
sample_males = complete[complete["Sex"] == "M"].copy()
sample_females = complete[complete["Sex"] == "F"].copy()
print("Самцов:", len(sample_males))
print("Самок:", len(sample_females))

<a id="s1-nature"></a>
<div style="font-family: 'Georgia', serif; border-left: 4px solid #d4af37; padding-left: 20px; margin: 40px 0 20px;">
    <div style="font-size: 12px; letter-spacing: 2px; color: #888;">РАЗДЕЛ 2</div>
    <h2 style="margin: 5px 0 0; color: #e0e0e0; font-weight: 400;">Природа события и генеральная совокупность</h2>
</div>

Из 134 строк с полными данными равновероятно выбирается одна строка о конкретном животном. Пока строка не выбрана — значение величины неизвестно, но каждой строке соответствует фиксированный набор измерений. Это и есть генеральная совокупность:

$$
\Omega = \{\omega_1,\ldots,\omega_{134}\}, \qquad
P(\omega_j) = \frac{1}{134}, \qquad j=1,\ldots,134.
$$

При равновероятном выборе строки масса тела и три длины имеют дискретное распределение с конечным числом значений:

$$
P(X=x_i)=p_i=\frac{n_i}{N},\qquad N=134.
$$

**Выборка с возвращением и без.** После взятия животного в выборку при отборе *с возвращением* оно остаётся доступным; при отборе *без возвращения* — исключается. Дисперсии выборочных средних различаются:

$$
\sigma_{\bar X,\;\text{с возвращением}}^{2}=\frac{\sigma^2}{n},
\qquad
\sigma_{\bar X,\;\text{без возвращения}}^{2}=\frac{\sigma^2}{n}\cdot\frac{N-n}{N-1}.
$$

<a id="s1-mass"></a>
<div style="font-family: 'Georgia', serif; border-left: 4px solid #d4af37; padding-left: 20px; margin: 40px 0 20px;">
    <div style="font-size: 12px; letter-spacing: 2px; color: #888;">РАЗДЕЛ 3</div>
    <h2 style="margin: 5px 0 0; color: #e0e0e0; font-weight: 400;">Величина 01 — масса тела</h2>
</div>

Случайная величина $X$ — масса тела животного в выбранной строке, $[\text{кг}]$:

$$
X(\omega_j) = m_j, \qquad
P(X=x_i)=p_i=\frac{n_i}{N},\quad p_i\ge 0,\quad \sum_{i=1}^{k}p_i=1.
$$

<details>
<summary style="cursor: pointer; color: #d4af37; font-weight: bold;">[ Показать/Скрыть код ]</summary>

In [ ]:
# === СТИЛЬ ГРАФИКОВ ===
import matplotlib as mpl
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(context="notebook", style="whitegrid")
mpl.rcParams.update({
    "figure.dpi": 110,
    "savefig.dpi": 140,
    "figure.facecolor": "#1a1a1a",
    "savefig.facecolor": "#1a1a1a",
    "axes.facecolor": "#222222",
    "axes.edgecolor": "#333333",
    "axes.labelcolor": "#e0e0e0",
    "axes.titlecolor": "#d4af37",
    "axes.titleweight": "bold",
    "axes.titlesize": 14,
    "axes.labelsize": 10,
    "axes.titlepad": 12,
    "axes.prop_cycle": mpl.cycler(color=["#d4af37", "#9b2226", "#666666"]),
    "grid.color": "#333333",
    "grid.linewidth": 0.5,
    "xtick.color": "#888888",
    "ytick.color": "#888888",
    "text.color": "#e0e0e0",
    "font.family": "sans-serif",
    "legend.facecolor": "#222222",
    "legend.edgecolor": "#333333",
    "legend.labelcolor": "#e0e0e0",
    "patch.edgecolor": "#1a1a1a",
})
PALETTE = {"primary": "#d4af37", "secondary": "#9b2226", "muted": "#666666"}

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

mass = complete["kg"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(x=mass, bins=20, kde=True, edgecolor="black", ax=axes[0])
axes[0].set_title("Распределение масс")
axes[0].set_xlabel("Масса, кг")
axes[0].set_ylabel("Количество значений")
sns.histplot(x=np.log10(mass), bins=20, kde=True, edgecolor="black", ax=axes[1])
axes[1].set_title("Распределение логарифма масс")
axes[1].set_xlabel("log₁₀(Масса/1кг)")
axes[1].set_ylabel("Количество значений")
axes[0].ticklabel_format(axis="x", style="plain", useOffset=False)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

mass = pd.to_numeric(selected["kg"].astype(str).str.replace(",", ".", regex=False), errors="coerce").dropna()
log_mass = np.log10(mass[mass > 0])
fig, ax = plt.subplots(figsize=(7, 5))
stats.probplot(log_mass, dist="norm", plot=ax)
ax.set_title("Q–Q-график логарифма массы")
ax.set_xlabel("Теоретические квантили нормального распределения")
ax.set_ylabel("Наблюдаемые значения log₁₀ массы")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
mass = complete["kg"]
mass_table = mass.value_counts().sort_index().reset_index()
mass_table.columns = ["mass_kg", "count"]
mass_table["p_i"] = mass_table["count"] / len(mass)
mass_table["P"] = mass_table["p_i"].cumsum()
print("Количество масс:", len(mass))
print("Количество уникальных масс:", len(mass_table))
print("Сумма относительных частот:", mass_table["p_i"].sum())
with pd.option_context("display.max_rows", None):
    display(mass_table)

In [ ]:
mass_table = mass_table.sort_values("mass_kg")
mass_table["P"] = mass_table["p_i"].cumsum()
fig, ax = plt.subplots(figsize=(7, 5))
ax.step(mass_table["mass_kg"], mass_table["P"], where="post")
ax.set_title("График функции распределения массы")
ax.set_xlabel("Масса, кг")
ax.set_ylabel("Накопленная доля")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
expected_mass = (mass_table["mass_kg"] * mass_table["p_i"]).sum()
variance_mass = ((mass_table["mass_kg"] - mass.mean()) ** 2 * mass_table["p_i"]).sum()
std_mass = np.sqrt(variance_mass)
modes_mass = mass.mode()
max_count = mass.value_counts().max()
print(f"Мат. ожидание по таблице частот, кг: {expected_mass:,.2f}")
print(f"Количество масс: {mass.count():,.2f}")
print(f"Сумма масс, кг: {mass.sum():.2f}")
print(f"Средняя масса, кг: {mass.mean():,.2f}")
print(f"Медиана массы, кг: {mass.median():,.2f}")
print(f"Минимальная масса, кг: {mass.min():,.2f}")
print(f"Максимальная масса, кг: {mass.max():,.2f}")
print(f"Cтандартное отклонение, кг: {mass.std(ddof=0):,.2f}")
print(f"Первый квартиль (25%), кг: {mass.quantile(0.25):,.2f}")
print(f"Третий квартиль (75%), кг: {mass.quantile(0.75):,.2f}")
print(f"95-й перцентиль, кг: {mass.quantile(0.95):,.2f}")
print(f"Дисперсия, кг²: {variance_mass:,.2f}")
print(f"Стандартное отклонение, кг: {std_mass:,.2f}")
print("Мода, кг:", modes_mass.tolist())
print("Число повторений каждой моды:", max_count)

mass_normalized = (mass - mass.min()) / (mass.max() - mass.min())
print("Минимум:", mass_normalized.min())
print("Максимум:", mass_normalized.max())

In [ ]:
rng = np.random.default_rng(42)
population = complete["kg"].to_numpy()
true_mean = population.mean()
sample_sizes = [10, 30, 100, 300, 1000, 3000]
repeats = 1000
results = []
for n in sample_sizes:
    samples = rng.choice(population, size=(repeats, n), replace=True)
    estimates = samples.mean(axis=1)
    errors = estimates - true_mean
    results.append({
        "Размер выборки": n,
        "Среднее оценок, кг": estimates.mean(),
        "RMSE, кг": np.sqrt(np.mean(errors ** 2)),
        "Нижняя граница 95% оценок, кг": np.quantile(estimates, 0.025),
        "Верхняя граница 95% оценок, кг": np.quantile(estimates, 0.975)
    })
consistency_mass = pd.DataFrame(results)
print(f"Средняя масса генеральной совокупности: {true_mean:.2f} кг")
display(consistency_mass)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(consistency_mass["Размер выборки"], consistency_mass["RMSE, кг"], marker="o", color=PALETTE["primary"])
plt.xscale("log")
plt.xlabel("Размер выборки, n")
plt.ylabel("RMSE среднего, кг")
plt.title("Ошибка оценки средней массы")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
mass_with = sample_with["kg"]
print("Количество:", mass_with.count())
print(f"Среднее, кг: {mass_with.mean():.2f}")
print(f"Медиана, кг: {mass_with.median():.2f}")
print(f"Минимум, кг: {mass_with.min():.2f}")
print(f"Максимум, кг: {mass_with.max():.2f}")
print(f"Стандартное отклонение, кг: {mass_with.std(ddof=0):.2f}")
print(f"Первый квартиль, кг: {mass_with.quantile(0.25):.2f}")
print(f"Третий квартиль, кг: {mass_with.quantile(0.75):.2f}")
print(f"95-й перцентиль, кг: {mass_with.quantile(0.95):.2f}")

In [ ]:
mass_without = sample_without["kg"]
print("Количество:", mass_without.count())
print(f"Среднее, кг: {mass_without.mean():.2f}")
print(f"Медиана, кг: {mass_without.median():.2f}")
print(f"Минимум, кг: {mass_without.min():.2f}")
print(f"Максимум, кг: {mass_without.max():.2f}")
print(f"Стандартное отклонение, кг: {mass_without.std(ddof=0):.2f}")
print(f"Первый квартиль, кг: {mass_without.quantile(0.25):.2f}")
print(f"Третий квартиль, кг: {mass_without.quantile(0.75):.2f}")
print(f"95-й перцентиль, кг: {mass_without.quantile(0.95):.2f}")

In [ ]:
mass_males = sample_males["kg"]
print("Количество:", mass_males.count())
print(f"Среднее, кг: {mass_males.mean():.2f}")
print(f"Медиана, кг: {mass_males.median():.2f}")
print(f"Минимум, кг: {mass_males.min():.2f}")
print(f"Максимум, кг: {mass_males.max():.2f}")
print(f"Стандартное отклонение, кг: {mass_males.std(ddof=0):.2f}")
print(f"Первый квартиль, кг: {mass_males.quantile(0.25):.2f}")
print(f"Третий квартиль, кг: {mass_males.quantile(0.75):.2f}")
print(f"95-й перцентиль, кг: {mass_males.quantile(0.95):.2f}")

In [ ]:
mass_females = sample_females["kg"]
print("Количество:", mass_females.count())
print(f"Среднее, кг: {mass_females.mean():.2f}")
print(f"Медиана, кг: {mass_females.median():.2f}")
print(f"Минимум, кг: {mass_females.min():.2f}")
print(f"Максимум, кг: {mass_females.max():.2f}")
print(f"Стандартное отклонение, кг: {mass_females.std(ddof=0):.2f}")
print(f"Первый квартиль, кг: {mass_females.quantile(0.25):.2f}")
print(f"Третий квартиль, кг: {mass_females.quantile(0.75):.2f}")
print(f"95-й перцентиль, кг: {mass_females.quantile(0.95):.2f}")

In [ ]:
print(f"Среднее генеральной совокупности, кг: {mass.mean():.2f}")
print(f"Среднее выборки с возвращением, кг: {mass_with.mean():.2f}")
print(f"Среднее выборки без возвращения, кг: {mass_without.mean():.2f}")
print(f"Ошибка среднего с возвращением, %: {abs(mass_with.mean() - mass.mean()) / mass.mean() * 100:.2f}")
print(f"Ошибка среднего без возвращения, %: {abs(mass_without.mean() - mass.mean()) / mass.mean() * 100:.2f}")

In [ ]:
difference = mass_males.mean() - mass_females.mean()
relative_difference = difference / mass_females.mean() * 100
print(f"Среднее всей совокупности, кг: {mass.mean():.2f}")
print(f"Среднее самцов, кг: {mass_males.mean():.2f}")
print(f"Среднее самок, кг: {mass_females.mean():.2f}")
print(f"Разность средних «самцы − самки», кг: {difference:.2f}")
print(f"Средняя масса тела самцов выше средней массы тела самок на {relative_difference:.2f}%")

</details>

<a id="s1-blength"></a>
<div style="font-family: 'Georgia', serif; border-left: 4px solid #d4af37; padding-left: 20px; margin: 40px 0 20px;">
    <div style="font-size: 12px; letter-spacing: 2px; color: #888;">РАЗДЕЛ 4</div>
    <h2 style="margin: 5px 0 0; color: #e0e0e0; font-weight: 400;">Величина 02 — длина тела</h2>
</div>

Случайная величина $Y$ — длина тела в выбранной строке, $[\text{мм}]$. По Q–Q-графику принимается гипотеза о распределении Фишера:

$$
Y(\omega_j) = b_j, \qquad P(Y=y_i)=p_i=\frac{n_i}{N}.
$$

<details>
<summary style="cursor: pointer; color: #d4af37; font-weight: bold;">[ Показать/Скрыть код ]</summary>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

b_length = complete["C-R curved"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(x=b_length, bins=20, kde=True, edgecolor="black", ax=axes[0])
axes[0].set_title("Распределение длин тела")
axes[0].set_xlabel("Длина, мм")
axes[0].set_ylabel("Количество значений")
sns.histplot(x=np.log10(b_length), bins=20, kde=True, edgecolor="black", ax=axes[1])
axes[1].set_title("Распределение логарифма длин тела")
axes[1].set_xlabel("log₁₀(Длина/1мм)")
axes[1].set_ylabel("Количество значений")
axes[0].ticklabel_format(axis="x", style="plain", useOffset=False)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

b_length = pd.to_numeric(selected["C-R curved"].astype(str).str.replace(",", ".", regex=False), errors="coerce").dropna()
params = stats.f.fit(b_length, floc=0)
fig, ax = plt.subplots(figsize=(7, 5))
stats.probplot(b_length, dist=stats.f, sparams=params, plot=ax)
ax.set_title("Q–Q-график длины тела: распределение Фишера")
ax.set_xlabel("Теоретические квантили распределения Фишера")
ax.set_ylabel("Длина тела, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
b_length = complete["C-R curved"]
b_length_table = b_length.value_counts().sort_index().reset_index()
b_length_table.columns = ["b_length_mm", "count"]
b_length_table["p_i"] = b_length_table["count"] / len(b_length)
b_length_table["P"] = b_length_table["p_i"].cumsum()
print("Количество длин тела:", len(b_length))
print("Количество уникальных длин тела:", len(b_length_table))
print("Сумма относительных частот:", b_length_table["p_i"].sum())
with pd.option_context("display.max_rows", None):
    display(b_length_table)

In [ ]:
b_length_table = b_length_table.sort_values("b_length_mm")
b_length_table["P"] = b_length_table["p_i"].cumsum()
fig, ax = plt.subplots(figsize=(7, 5))
ax.step(b_length_table["b_length_mm"], b_length_table["P"], where="post")
ax.set_title("График функции распределения длины тел")
ax.set_xlabel("Длина, мм")
ax.set_ylabel("Накопленная доля")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
expected_b_length = (b_length_table["b_length_mm"] * b_length_table["p_i"]).sum()
variance_b_length = ((b_length_table["b_length_mm"] - b_length.mean()) ** 2 * b_length_table["p_i"]).sum()
std_b_length = np.sqrt(variance_b_length)
modes_b_length = b_length.mode()
max_count = b_length.value_counts().max()
print(f"Мат. ожидание по таблице частот, мм: {expected_b_length:,.2f}")
print(f"Количество длин тел: {b_length.count():,.2f}")
print(f"Сумма длин тел, мм: {b_length.sum():.2f}")
print(f"Средняя длина тел, мм: {b_length.mean():,.2f}")
print(f"Медиана длин тел, мм: {b_length.median():,.2f}")
print(f"Минимальная длина тел, мм: {b_length.min():,.2f}")
print(f"Максимальная длина тел, мм: {b_length.max():,.2f}")
print(f"Cтандартное отклонение, мм: {b_length.std(ddof=0):,.2f}")
print(f"Первый квартиль (25%), мм: {b_length.quantile(0.25):,.2f}")
print(f"Третий квартиль (75%), мм: {b_length.quantile(0.75):,.2f}")
print(f"95-й перцентиль, мм: {b_length.quantile(0.95):,.2f}")
print(f"Дисперсия, мм²: {variance_b_length:,.2f}")
print(f"Стандартное отклонение, мм: {std_b_length:,.2f}")
print("Мода, мм:", modes_b_length.tolist())
print("Число повторений каждой моды:", max_count)

In [ ]:
rng = np.random.default_rng(42)
population = complete["C-R curved"].to_numpy()
true_mean = population.mean()
sample_sizes = [10, 30, 100, 300, 1000, 3000]
repeats = 100
results = []
for n in sample_sizes:
    samples = rng.choice(population, size=(repeats, n), replace=True)
    estimates = samples.mean(axis=1)
    errors = estimates - true_mean
    results.append({
        "Размер выборки": n,
        "Среднее оценок, мм": estimates.mean(),
        "RMSE, мм": np.sqrt(np.mean(errors ** 2)),
        "Нижняя граница 95% оценок, мм": np.quantile(estimates, 0.025),
        "Верхняя граница 95% оценок, мм": np.quantile(estimates, 0.975)
    })
consistency_b_length = pd.DataFrame(results)
print(f"Средняя длина тела в генеральной совокупности: {true_mean:.2f} мм")
display(consistency_b_length)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(consistency_b_length["Размер выборки"], consistency_b_length["RMSE, мм"], marker="o", color=PALETTE["primary"])
plt.xscale("log")
plt.xlabel("Размер выборки, n")
plt.ylabel("RMSE среднего, мм")
plt.title("Ошибка оценки длины тел")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
b_with = sample_with["C-R curved"]
print("Количество:", b_with.count())
print(f"Среднее, мм: {b_with.mean():.2f}")
print(f"Медиана, мм: {b_with.median():.2f}")
print(f"Минимум, мм: {b_with.min():.2f}")
print(f"Максимум, мм: {b_with.max():.2f}")
print(f"Стандартное отклонение, мм: {b_with.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {b_with.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {b_with.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {b_with.quantile(0.95):.2f}")

In [ ]:
b_without = sample_without["C-R curved"]
print("Количество:", b_without.count())
print(f"Среднее, мм: {b_without.mean():.2f}")
print(f"Медиана, мм: {b_without.median():.2f}")
print(f"Минимум, мм: {b_without.min():.2f}")
print(f"Максимум, мм: {b_without.max():.2f}")
print(f"Стандартное отклонение, мм: {b_without.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {b_without.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {b_without.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {b_without.quantile(0.95):.2f}")

In [ ]:
b_males = sample_males["C-R curved"]
print("Количество:", b_males.count())
print(f"Среднее, мм: {b_males.mean():.2f}")
print(f"Медиана, мм: {b_males.median():.2f}")
print(f"Минимум, мм: {b_males.min():.2f}")
print(f"Максимум, мм: {b_males.max():.2f}")
print(f"Стандартное отклонение, мм: {b_males.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {b_males.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {b_males.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {b_males.quantile(0.95):.2f}")

In [ ]:
b_females = sample_females["C-R curved"]
print("Количество:", b_females.count())
print(f"Среднее, мм: {b_females.mean():.2f}")
print(f"Медиана, мм: {b_females.median():.2f}")
print(f"Минимум, мм: {b_females.min():.2f}")
print(f"Максимум, мм: {b_females.max():.2f}")
print(f"Стандартное отклонение, мм: {b_females.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {b_females.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {b_females.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {b_females.quantile(0.95):.2f}")

In [ ]:
print(f"Среднее генеральной совокупности, мм: {b_length.mean():.2f}")
print(f"Среднее выборки с возвращением, мм: {b_with.mean():.2f}")
print(f"Среднее выборки без возвращения, мм: {b_without.mean():.2f}")
print(f"Ошибка среднего с возвращением, %: {abs(b_with.mean() - b_length.mean()) / b_length.mean() * 100:.2f}")
print(f"Ошибка среднего без возвращения, %: {abs(b_without.mean() - b_length.mean()) / b_length.mean() * 100:.2f}")

In [ ]:
difference = b_males.mean() - b_females.mean()
relative_difference = difference / b_females.mean() * 100
print(f"Среднее всей совокупности, мм: {b_length.mean():.2f}")
print(f"Среднее самцов, мм: {b_males.mean():.2f}")
print(f"Среднее самок, мм: {b_females.mean():.2f}")
print(f"Разность средних «самцы − самки», мм: {difference:.2f}")
print(f"Средняя длина самцов выше средней длины самок на {relative_difference:.2f}%")

</details>

<a id="s1-humerus"></a>
<div style="font-family: 'Georgia', serif; border-left: 4px solid #d4af37; padding-left: 20px; margin: 40px 0 20px;">
    <div style="font-size: 12px; letter-spacing: 2px; color: #888;">РАЗДЕЛ 5</div>
    <h2 style="margin: 5px 0 0; color: #e0e0e0; font-weight: 400;">Величина 03 — длина плечевой кости</h2>
</div>

Случайная величина $Z$ — длина плечевой кости, $[\text{мм}]$. Гипотеза о распределении Фишера подтверждается Q–Q-графиком:

$$
Z(\omega_j) = h_j, \qquad P(Z=z_i)=p_i=\frac{n_i}{N}.
$$

<details>
<summary style="cursor: pointer; color: #d4af37; font-weight: bold;">[ Показать/Скрыть код ]</summary>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

h_length = complete["R humeral max"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(x=h_length, bins=20, kde=True, edgecolor="black", ax=axes[0])
axes[0].set_title("Распределение длин плечевых костей")
axes[0].set_xlabel("Длина, мм")
axes[0].set_ylabel("Количество значений")
sns.histplot(x=np.log10(h_length), bins=20, kde=True, edgecolor="black", ax=axes[1])
axes[1].set_title("Распределение логарифма длин плечевых костей")
axes[1].set_xlabel("log₁₀(Длина/1мм)")
axes[1].set_ylabel("Количество значений")
axes[0].ticklabel_format(axis="x", style="plain", useOffset=False)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

h_length = pd.to_numeric(selected["R humeral max"].astype(str).str.replace(",", ".", regex=False), errors="coerce").dropna()
params = stats.f.fit(h_length, floc=0)
fig, ax = plt.subplots(figsize=(7, 5))
stats.probplot(h_length, dist=stats.f, sparams=params, plot=ax)
ax.set_title("Q–Q-график длины плечевой кости: распределение Фишера")
ax.set_xlabel("Теоретические квантили распределения Фишера")
ax.set_ylabel("Длина плечевой кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
h_length = complete["R humeral max"]
h_length_table = h_length.value_counts().sort_index().reset_index()
h_length_table.columns = ["h_length_mm", "count"]
h_length_table["p_i"] = h_length_table["count"] / len(h_length)
h_length_table["P"] = h_length_table["p_i"].cumsum()
print("Количество длин плечевых костей:", len(h_length))
print("Количество уникальных длин плечевых костей:", len(h_length_table))
print("Сумма относительных частот:", h_length_table["p_i"].sum())
with pd.option_context("display.max_rows", None):
    display(h_length_table)

In [ ]:
h_length_table = h_length_table.sort_values("h_length_mm")
h_length_table["P"] = h_length_table["p_i"].cumsum()
fig, ax = plt.subplots(figsize=(7, 5))
ax.step(h_length_table["h_length_mm"], h_length_table["P"], where="post")
ax.set_title("График функции распределения длины плечевых костей")
ax.set_xlabel("Длина, мм")
ax.set_ylabel("Накопленная доля")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
expected_h_length = (h_length_table["h_length_mm"] * h_length_table["p_i"]).sum()
variance_h_length = ((h_length_table["h_length_mm"] - h_length.mean()) ** 2 * h_length_table["p_i"]).sum()
std_h_length = np.sqrt(variance_h_length)
modes_h_length = h_length.mode()
max_count = h_length.value_counts().max()
print(f"Мат. ожидание по таблице частот, мм: {expected_h_length:,.2f}")
print(f"Количество длин плечевых костей: {h_length.count():,.2f}")
print(f"Сумма длин плечевых костей, мм: {h_length.sum():.2f}")
print(f"Средняя длина плечевых костей, мм: {h_length.mean():,.2f}")
print(f"Медиана длин плечевых костей, мм: {h_length.median():,.2f}")
print(f"Минимальная длина плечевых костей, мм: {h_length.min():,.2f}")
print(f"Максимальная длина плечевых костей, мм: {h_length.max():,.2f}")
print(f"Cтандартное отклонение, мм: {h_length.std(ddof=0):,.2f}")
print(f"Первый квартиль (25%), мм: {h_length.quantile(0.25):,.2f}")
print(f"Третий квартиль (75%), мм: {h_length.quantile(0.75):,.2f}")
print(f"95-й перцентиль, мм: {h_length.quantile(0.95):,.2f}")
print(f"Дисперсия, мм²: {variance_h_length:,.2f}")
print(f"Стандартное отклонение, мм: {std_h_length:,.2f}")
print("Мода, мм:", modes_h_length.tolist())
print("Число повторений каждой моды:", max_count)

In [ ]:
rng = np.random.default_rng(42)
population = complete["R humeral max"].to_numpy()
true_mean = population.mean()
sample_sizes = [10, 30, 100, 300, 1000, 3000]
repeats = 100
results = []
for n in sample_sizes:
    samples = rng.choice(population, size=(repeats, n), replace=True)
    estimates = samples.mean(axis=1)
    errors = estimates - true_mean
    results.append({
        "Размер выборки": n,
        "Среднее оценок, мм": estimates.mean(),
        "RMSE, мм": np.sqrt(np.mean(errors ** 2)),
        "Нижняя граница 95% оценок, мм": np.quantile(estimates, 0.025),
        "Верхняя граница 95% оценок, мм": np.quantile(estimates, 0.975)
    })
consistency_h_length = pd.DataFrame(results)
print(f"Средняя длина плечевой кости в генеральной совокупности: {true_mean:.2f} мм")
display(consistency_h_length)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(consistency_h_length["Размер выборки"], consistency_h_length["RMSE, мм"], marker="o", color=PALETTE["primary"])
plt.xscale("log")
plt.xlabel("Размер выборки, n")
plt.ylabel("RMSE среднего, мм")
plt.title("Ошибка оценки длины плечевой кости")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
h_with = sample_with["R humeral max"]
print("Количество:", h_with.count())
print(f"Среднее, мм: {h_with.mean():.2f}")
print(f"Медиана, мм: {h_with.median():.2f}")
print(f"Минимум, мм: {h_with.min():.2f}")
print(f"Максимум, мм: {h_with.max():.2f}")
print(f"Стандартное отклонение, мм: {h_with.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {h_with.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {h_with.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {h_with.quantile(0.95):.2f}")

In [ ]:
h_without = sample_without["R humeral max"]
print("Количество:", h_without.count())
print(f"Среднее, мм: {h_without.mean():.2f}")
print(f"Медиана, мм: {h_without.median():.2f}")
print(f"Минимум, мм: {h_without.min():.2f}")
print(f"Максимум, мм: {h_without.max():.2f}")
print(f"Стандартное отклонение, мм: {h_without.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {h_without.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {h_without.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {h_without.quantile(0.95):.2f}")

In [ ]:
h_males = sample_males["R humeral max"]
print("Количество:", h_males.count())
print(f"Среднее, мм: {h_males.mean():.2f}")
print(f"Медиана, мм: {h_males.median():.2f}")
print(f"Минимум, мм: {h_males.min():.2f}")
print(f"Максимум, мм: {h_males.max():.2f}")
print(f"Стандартное отклонение, мм: {h_males.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {h_males.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {h_males.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {h_males.quantile(0.95):.2f}")

In [ ]:
h_females = sample_females["R humeral max"]
print("Количество:", h_females.count())
print(f"Среднее, мм: {h_females.mean():.2f}")
print(f"Медиана, мм: {h_females.median():.2f}")
print(f"Минимум, мм: {h_females.min():.2f}")
print(f"Максимум, мм: {h_females.max():.2f}")
print(f"Стандартное отклонение, мм: {h_females.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {h_females.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {h_females.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {h_females.quantile(0.95):.2f}")

In [ ]:
print(f"Среднее генеральной совокупности, мм: {h_length.mean():.2f}")
print(f"Среднее выборки с возвращением, мм: {h_with.mean():.2f}")
print(f"Среднее выборки без возвращения, мм: {h_without.mean():.2f}")
print(f"Ошибка среднего с возвращением, %: {abs(h_with.mean() - h_length.mean()) / h_length.mean() * 100:.2f}")
print(f"Ошибка среднего без возвращения, %: {abs(h_without.mean() - h_length.mean()) / h_length.mean() * 100:.2f}")

In [ ]:
difference = h_males.mean() - h_females.mean()
relative_difference = difference / h_females.mean() * 100
print(f"Среднее всей совокупности, мм: {h_length.mean():.2f}")
print(f"Среднее самцов, мм: {h_males.mean():.2f}")
print(f"Среднее самок, мм: {h_females.mean():.2f}")
print(f"Разность средних «самцы − самки», мм: {difference:.2f}")
print(f"Средняя длина плечевых костей самцов выше средней длины плечевых костей самок на {relative_difference:.2f}%")

</details>

<a id="s1-femur"></a>
<div style="font-family: 'Georgia', serif; border-left: 4px solid #d4af37; padding-left: 20px; margin: 40px 0 20px;">
    <div style="font-size: 12px; letter-spacing: 2px; color: #888;">РАЗДЕЛ 6</div>
    <h2 style="margin: 5px 0 0; color: #e0e0e0; font-weight: 400;">Величина 04 — длина бедренной кости</h2>
</div>

Случайная величина $W$ — длина бедренной кости, $[\text{мм}]$:

$$
W(\omega_j) = f_j, \qquad P(W=w_i)=p_i=\frac{n_i}{N}.
$$

<details>
<summary style="cursor: pointer; color: #d4af37; font-weight: bold;">[ Показать/Скрыть код ]</summary>

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

f_length = complete["R femoral max"]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.histplot(x=f_length, bins=20, kde=True, edgecolor="black", ax=axes[0])
axes[0].set_title("Распределение длин бедренных костей")
axes[0].set_xlabel("Длина, мм")
axes[0].set_ylabel("Количество значений")
sns.histplot(x=np.log10(f_length), bins=20, kde=True, edgecolor="black", ax=axes[1])
axes[1].set_title("Распределение логарифма длин бедренных костей")
axes[1].set_xlabel("log₁₀(Длина/1мм)")
axes[1].set_ylabel("Количество значений")
axes[0].ticklabel_format(axis="x", style="plain", useOffset=False)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

f_length = pd.to_numeric(selected["R femoral max"].astype(str).str.replace(",", ".", regex=False), errors="coerce").dropna()
params = stats.f.fit(f_length, floc=0)
fig, ax = plt.subplots(figsize=(7, 5))
stats.probplot(f_length, dist=stats.f, sparams=params, plot=ax)
ax.set_title("Q–Q-график длины бедренной кости: распределение Фишера")
ax.set_xlabel("Теоретические квантили распределения Фишера")
ax.set_ylabel("Длина бедренной кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
f_length = complete["R femoral max"]
f_length_table = f_length.value_counts().sort_index().reset_index()
f_length_table.columns = ["f_length_mm", "count"]
f_length_table["p_i"] = f_length_table["count"] / len(f_length)
f_length_table["P"] = f_length_table["p_i"].cumsum()
print("Количество длин бедренных костей:", len(f_length))
print("Количество уникальных длин бедренных костей:", len(f_length_table))
print("Сумма относительных частот:", f_length_table["p_i"].sum())
with pd.option_context("display.max_rows", None):
    display(f_length_table)

In [ ]:
f_length_table = f_length_table.sort_values("f_length_mm")
f_length_table["P"] = f_length_table["p_i"].cumsum()
fig, ax = plt.subplots(figsize=(7, 5))
ax.step(f_length_table["f_length_mm"], f_length_table["P"], where="post")
ax.set_title("График функции распределения длины бедренных костей")
ax.set_xlabel("Длина, мм")
ax.set_ylabel("Накопленная доля")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
expected_f_length = (f_length_table["f_length_mm"] * f_length_table["p_i"]).sum()
variance_f_length = ((f_length_table["f_length_mm"] - f_length.mean()) ** 2 * f_length_table["p_i"]).sum()
std_f_length = np.sqrt(variance_f_length)
modes_f_length = f_length.mode()
max_count = f_length.value_counts().max()
print(f"Мат. ожидание по таблице частот, мм: {expected_f_length:,.2f}")
print(f"Сумма длин бедренных костей, мм: {f_length.sum():.2f}")
print(f"Средняя длина бедренных костей, мм: {f_length.mean():,.2f}")
print(f"Медиана длин бедренных костей, мм: {f_length.median():,.2f}")
print(f"Минимальная длина бедренных костей, мм: {f_length.min():,.2f}")
print(f"Максимальная длина бедренных костей, мм: {f_length.max():,.2f}")
print(f"Cтандартное отклонение, мм: {f_length.std(ddof=0):,.2f}")
print(f"Первый квартиль (25%), мм: {f_length.quantile(0.25):,.2f}")
print(f"Третий квартиль (75%), мм: {f_length.quantile(0.75):,.2f}")
print(f"95-й перцентиль, мм: {f_length.quantile(0.95):,.2f}")
print(f"Дисперсия, мм²: {variance_f_length:,.2f}")
print(f"Стандартное отклонение, мм: {std_f_length:,.2f}")
print("Мода, мм:", modes_f_length.tolist())
print("Число повторений каждой моды:", max_count)

In [ ]:
rng = np.random.default_rng(42)
population = complete["R femoral max"].to_numpy()
true_mean = population.mean()
sample_sizes = [10, 30, 100, 300, 1000, 3000]
repeats = 1000
results = []
for n in sample_sizes:
    samples = rng.choice(population, size=(repeats, n), replace=True)
    estimates = samples.mean(axis=1)
    errors = estimates - true_mean
    results.append({
        "Размер выборки": n,
        "Среднее оценок, мм": estimates.mean(),
        "RMSE, мм": np.sqrt(np.mean(errors ** 2)),
        "Нижняя граница 95% оценок, мм": np.quantile(estimates, 0.025),
        "Верхняя граница 95% оценок, мм": np.quantile(estimates, 0.975)
    })
consistency_f_length = pd.DataFrame(results)
print(f"Средняя длина бедренной кости в генеральной совокупности: {true_mean:.2f} мм")
display(consistency_f_length)

In [ ]:
plt.figure(figsize=(7, 4))
plt.plot(consistency_f_length["Размер выборки"], consistency_f_length["RMSE, мм"], marker="o", color=PALETTE["primary"])
plt.xscale("log")
plt.xlabel("Размер выборки, n")
plt.ylabel("RMSE среднего, мм")
plt.title("Ошибка оценки средней длины бедренной кости")
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
f_with = sample_with["R femoral max"]
print("Количество:", f_with.count())
print(f"Среднее, мм: {f_with.mean():.2f}")
print(f"Медиана, мм: {f_with.median():.2f}")
print(f"Минимум, мм: {f_with.min():.2f}")
print(f"Максимум, мм: {f_with.max():.2f}")
print(f"Стандартное отклонение, мм: {f_with.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {f_with.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {f_with.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {f_with.quantile(0.95):.2f}")

In [ ]:
f_without = sample_without["R femoral max"]
print("Количество:", f_without.count())
print(f"Среднее, мм: {f_without.mean():.2f}")
print(f"Медиана, мм: {f_without.median():.2f}")
print(f"Минимум, мм: {f_without.min():.2f}")
print(f"Максимум, мм: {f_without.max():.2f}")
print(f"Стандартное отклонение, мм: {f_without.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {f_without.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {f_without.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {f_without.quantile(0.95):.2f}")

In [ ]:
f_males = sample_males["R femoral max"]
print("Количество:", f_males.count())
print(f"Среднее, мм: {f_males.mean():.2f}")
print(f"Медиана, мм: {f_males.median():.2f}")
print(f"Минимум, мм: {f_males.min():.2f}")
print(f"Максимум, мм: {f_males.max():.2f}")
print(f"Стандартное отклонение, мм: {f_males.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {f_males.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {f_males.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {f_males.quantile(0.95):.2f}")

In [ ]:
f_females = sample_females["R femoral max"]
print("Количество:", f_females.count())
print(f"Среднее, мм: {f_females.mean():.2f}")
print(f"Медиана, мм: {f_females.median():.2f}")
print(f"Минимум, мм: {f_females.min():.2f}")
print(f"Максимум, мм: {f_females.max():.2f}")
print(f"Стандартное отклонение, мм: {f_females.std(ddof=0):.2f}")
print(f"Первый квартиль, мм: {f_females.quantile(0.25):.2f}")
print(f"Третий квартиль, мм: {f_females.quantile(0.75):.2f}")
print(f"95-й перцентиль, мм: {f_females.quantile(0.95):.2f}")

In [ ]:
print(f"Среднее генеральной совокупности, мм: {f_length.mean():.2f}")
print(f"Среднее выборки с возвращением, мм: {f_with.mean():.2f}")
print(f"Среднее выборки без возвращения, мм: {f_without.mean():.2f}")
print(f"Ошибка среднего с возвращением, %: {abs(f_with.mean() - f_length.mean()) / f_length.mean() * 100:.2f}")
print(f"Ошибка среднего без возвращения, %: {abs(f_without.mean() - f_length.mean()) / f_length.mean() * 100:.2f}")

In [ ]:
difference = f_males.mean() - f_females.mean()
relative_difference = difference / f_females.mean() * 100
print(f"Среднее всей совокупности, мм: {f_length.mean():.2f}")
print(f"Среднее самцов, мм: {f_males.mean():.2f}")
print(f"Среднее самок, мм: {f_females.mean():.2f}")
print(f"Разность средних «самцы − самки», мм: {difference:.2f}")
print(f"Средняя длина бедренных костей самцов выше средней длины бедренных костей самок на {relative_difference:.2f}%")

</details>

<a id="s1-deps"></a>
<div style="font-family: 'Georgia', serif; border-left: 4px solid #d4af37; padding-left: 20px; margin: 40px 0 20px;">
    <div style="font-size: 12px; letter-spacing: 2px; color: #888;">РАЗДЕЛ 7</div>
    <h2 style="margin: 5px 0 0; color: #e0e0e0; font-weight: 400;">Зависимости и нормализация</h2>
</div>

Попарные диаграммы рассеяния. Если величины независимы, то для событий $A=\{X\le\text{med}_X\}$ и $B=\{Y\le\text{med}_Y\}$ должно выполняться $P(A\cap B)=P(A)\,P(B)$.

<details>
<summary style="cursor: pointer; color: #d4af37; font-weight: bold;">[ Показать/Скрыть код ]</summary>

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(mass, b_length, alpha=0.6, s=20)
ax.set_title("Масса и длина тела")
ax.set_xlabel("Масса, кг")
ax.set_ylabel("Длина тела, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(mass, h_length, alpha=0.6, s=20)
ax.set_title("Масса тела и длина плечевой кости")
ax.set_xlabel("Масса, кг")
ax.set_ylabel("Длина плечевой кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(mass, f_length, alpha=0.6, s=20)
ax.set_title("Масса тела и длина бедренной кости")
ax.set_xlabel("Масса, кг")
ax.set_ylabel("Длина бедренной кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(b_length, h_length, alpha=0.6, s=20)
ax.set_title("Длина тела и длина плечевой кости")
ax.set_xlabel("Длина тела, мм")
ax.set_ylabel("Длина плечевой кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(b_length, f_length, alpha=0.6, s=20)
ax.set_title("Длина тела и длина бедренной кости")
ax.set_xlabel("Длина тела, мм")
ax.set_ylabel("Длина бедренной кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(h_length, f_length, alpha=0.6, s=20)
ax.set_title("Длина плечевой и длина бедренной костей")
ax.set_xlabel("Длина плечевой кости, мм")
ax.set_ylabel("Длина бедренной кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ancestry = complete["Chinese Ancestry"]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ancestry, mass, alpha=0.6, s=20)
ax.set_title("Масса тела и доля китайского происхождения")
ax.set_xlabel("Доля китайского происхождения")
ax.set_ylabel("Масса тела, кг")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ancestry = complete["Chinese Ancestry"]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ancestry, b_length, alpha=0.6, s=20)
ax.set_title("Длина тела и доля китайского происхождения")
ax.set_xlabel("Доля китайского происхождения")
ax.set_ylabel("Длина тела, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ancestry = complete["Chinese Ancestry"]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ancestry, h_length, alpha=0.6, s=20)
ax.set_title("Длина плечевой кости и доля китайского происхождения")
ax.set_xlabel("Доля китайского происхождения")
ax.set_ylabel("Длина плечевой кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
ancestry = complete["Chinese Ancestry"]
fig, ax = plt.subplots(figsize=(7, 5))
ax.scatter(ancestry, f_length, alpha=0.6, s=20)
ax.set_title("Длина бедренной кости и доля китайского происхождения")
ax.set_xlabel("Доля китайского происхождения")
ax.set_ylabel("Длина бедренной кости, мм")
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
mass = complete["kg"]
b_length = complete["C-R curved"]
h_length = complete["R humeral max"]
f_length = complete["R femoral max"]
ancestry = complete["Chinese Ancestry"]
N = len(complete)
pairs = [
    ("Масса и длина тела", mass, b_length),
    ("Масса тела и длина плечевой кости", mass, h_length),
    ("Масса тела и длина бедренной кости", mass, f_length),
    ("Длина тела и длина плечевой кости", b_length, h_length),
    ("Длина тела и длина бедренной кости", b_length, f_length),
    ("Длина плечевой и длина бедренной костей", h_length, f_length),
    ("Масса тела и доля китайского происхождения", mass, ancestry),
    ("Длина тела и доля китайского происхождения", b_length, ancestry),
    ("Длина плечевой кости и доля китайского происхождения", h_length, ancestry),
    ("Длина бедренной кости и доля китайского происхождения", f_length, ancestry)
]
for name, first, second in pairs:
    assert first.notna().all() and second.notna().all()
    A = first <= first.median()
    B = second <= second.median()
    n_A = int(A.sum())
    n_B = int(B.sum())
    n_AB = int((A & B).sum())
    print(name)
    print("Порог первой величины:", first.median())
    print("Порог второй величины:", second.median())
    print(f"P(A): {n_A / N:.4f}")
    print(f"P(B): {n_B / N:.4f}")
    print(f"P(A ∩ B): {n_AB / N:.4f}")
    print(f"P(A) · P(B): {(n_A / N) * (n_B / N):.4f}")
    if n_AB * N == n_A * n_B:
        print("Для этих событий равенство выполнено.")
        print("Этого недостаточно для доказательства независимости величин.")
    else:
        print("Для выбранных событий равенство вероятностей нарушено.")
        print("Это указывает на зависимость в модели данной таблицы, но не определяет её силу и достоверность для других макак данного вида.")

    print()

In [ ]:
import matplotlib.pyplot as plt

data = complete[["kg", "C-R curved", "R humeral max", "R femoral max"]].copy()
normalized = (data - data.min()) / (data.max() - data.min())
normalized.columns = [
    "Масса",
    "Длина тела",
    "Плечевая кость",
    "Бедренная кость"
]
normalized.boxplot(figsize=(9, 5), whis=(0, 100))
plt.title("Сравнение признаков после min–max-нормализации")
plt.ylabel("Нормализованное значение")
plt.ylim(-0.05, 1.05)
plt.grid(axis="y", alpha=0.3)
plt.tight_layout()
plt.show()

</details>

<a id="s1-resume"></a>
<div style="font-family: 'Georgia', serif; border-left: 4px solid #2e9e6b; padding-left: 20px; margin: 40px 0 20px;">
    <div style="font-size: 12px; letter-spacing: 2px; color: #888;">РАЗДЕЛ 8</div>
    <h2 style="margin: 5px 0 0; color: #e0e0e0; font-weight: 400;">Выводы</h2>
</div>

- Распределение массы тела согласуется с логнормальным (подтверждено Q–Q-графиком логарифма массы).
- Распределения длины тела, длины плечевой и бедренной костей согласуются с распределением Фишера (Q–Q-графики).
- Для всех четырёх величин средние, медианы и стандартные отклонения в выборке **с возвращением** ближе к параметрам генеральной совокупности, чем в выборке без возвращения.
- Оценка среднего состоятельна: RMSE убывает с ростом объёма выборки.
- Все четыре величины попарно зависимы (нарушено равенство $P(A\cap B)=P(A)P(B)$ для событий «ниже медианы»).
- Все четыре величины зависят от пола (у самцов средние выше) и от доли китайского происхождения (наиболее выражено для массы тела).
- После min–max-нормализации медианы всех четырёх величин лежат ниже середины диапазона — распределения имеют правую асимметрию.